# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident.

In [ ]:
# loading relevant packages
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline
sns.set_style('whitegrid')

## Exploratory Data Analysis
- Load in the cleaned data

In [ ]:
df = pd.read_csv('data/AviationData_Cleaned.csv', low_memory=False)
print('Shape:', df.shape)
df.head()

## Explore safety metrics across models/makes
- Separate small planes (<=20 passengers) from large planes (>20 passengers)

In [ ]:
df_small = df[df['Total.Passengers'] <= 20].copy()
df_large = df[df['Total.Passengers'] > 20].copy()

print(f'Small planes (<=20 passengers): {df_small.shape[0]} records')
print(f'Large planes (>20 passengers): {df_large.shape[0]} records')

#### Analyzing Makes

Explore the human injury risk profile for small and larger Makes:
- Choose the 15 makes for each group with the lowest mean fatal/seriously injured fraction
- Plot the mean fatal/seriously injured fraction side-by-side

In [ ]:
# Calculate mean injury fraction by Make for small planes (min 10 records)
small_make_stats = (
    df_small.groupby('Make')
    .agg(mean_injury=('Fatal.Serious.Fraction', 'mean'), count=('Make', 'count'))
    .query('count >= 10')
    .sort_values('mean_injury')
    .head(15)
)

# Calculate mean injury fraction by Make for large planes (min 10 records)
large_make_stats = (
    df_large.groupby('Make')
    .agg(mean_injury=('Fatal.Serious.Fraction', 'mean'), count=('Make', 'count'))
    .query('count >= 10')
    .sort_values('mean_injury')
    .head(15)
)

# Plot side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.barh(small_make_stats.index, small_make_stats['mean_injury'], color='steelblue')
ax1.set_xlabel('Mean Fatal/Serious Injury Fraction')
ax1.set_title('Top 15 Safest Makes - Small Planes (<=20 passengers)')
ax1.invert_yaxis()

if len(large_make_stats) > 0:
    ax2.barh(large_make_stats.index, large_make_stats['mean_injury'], color='coral')
    ax2.set_xlabel('Mean Fatal/Serious Injury Fraction')
    ax2.set_title('Top 15 Safest Makes - Large Planes (>20 passengers)')
    ax2.invert_yaxis()
else:
    ax2.text(0.5, 0.5, 'Insufficient large plane data\n(fewer than 10 records per make)', 
             ha='center', va='center', transform=ax2.transAxes, fontsize=12)
    ax2.set_title('Large Planes - Insufficient Data')

plt.tight_layout()
plt.show()

print('Small plane make stats:')
print(small_make_stats)

**Distribution of injury rates: small makes**

Use a violinplot to look at the distribution of the fraction of passengers seriously/fatally injured for small airplane makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
top10_small_makes = small_make_stats.head(10).index.tolist()
df_small_top10 = df_small[df_small['Make'].isin(top10_small_makes)]

fig, ax = plt.subplots(figsize=(14, 6))
sns.violinplot(data=df_small_top10, x='Make', y='Fatal.Serious.Fraction', 
               order=top10_small_makes, ax=ax, palette='Blues')
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.set_title('Distribution of Fatal/Serious Injury Fraction - Top 10 Safest Small Makes')
ax.set_xlabel('Make')
ax.set_ylabel('Fatal/Serious Injury Fraction')
plt.tight_layout()
plt.show()

**Distribution of injury rates: large makes**

Use a stripplot for large airplane makes with the ten lowest mean serious/fatal injury rates.

In [ ]:
if len(large_make_stats) >= 1:
    top10_large_makes = large_make_stats.head(10).index.tolist()
    df_large_top10 = df_large[df_large['Make'].isin(top10_large_makes)]

    fig, ax = plt.subplots(figsize=(14, 6))
    sns.stripplot(data=df_large_top10, x='Make', y='Fatal.Serious.Fraction',
                  order=top10_large_makes, ax=ax, palette='Reds', jitter=True, alpha=0.7)
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    ax.set_title('Distribution of Fatal/Serious Injury Fraction - Top 10 Safest Large Makes')
    ax.set_xlabel('Make')
    ax.set_ylabel('Fatal/Serious Injury Fraction')
    plt.tight_layout()
    plt.show()
else:
    print('Not enough large plane makes with >= 10 records for a meaningful stripplot.')

**Evaluate the rate of aircraft destruction for both small and large aircraft by Make.**

Sort results and keep the lowest 15.

In [ ]:
# Destruction rate for small planes
small_destroyed = (
    df_small.groupby('Make')
    .agg(destroyed_rate=('Destroyed', 'mean'), count=('Make', 'count'))
    .query('count >= 10')
    .sort_values('destroyed_rate')
    .head(15)
)

# Destruction rate for large planes
large_destroyed = (
    df_large.groupby('Make')
    .agg(destroyed_rate=('Destroyed', 'mean'), count=('Make', 'count'))
    .query('count >= 10')
    .sort_values('destroyed_rate')
    .head(15)
)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(18, 6))

ax1.barh(small_destroyed.index, small_destroyed['destroyed_rate'], color='steelblue')
ax1.set_xlabel('Destruction Rate')
ax1.set_title('Top 15 Lowest Destruction Rate - Small Planes')
ax1.invert_yaxis()

if len(large_destroyed) > 0:
    ax2.barh(large_destroyed.index, large_destroyed['destroyed_rate'], color='coral')
    ax2.set_xlabel('Destruction Rate')
    ax2.set_title('Top 15 Lowest Destruction Rate - Large Planes')
    ax2.invert_yaxis()
else:
    ax2.text(0.5, 0.5, 'Insufficient large plane data', 
             ha='center', va='center', transform=ax2.transAxes)

plt.tight_layout()
plt.show()

print('Small plane destruction rates:')
print(small_destroyed)
if len(large_destroyed) > 0:
    print('\nLarge plane destruction rates:')
    print(large_destroyed)

#### Discussion: Findings for Makes

**Small Planes:**
- Maule and Aviat Aircraft stand out as the safest small plane makes, with the lowest mean fatal/serious injury fractions.
- Boeing also appears in the small plane dataset (likely twin-engine small jets), with a very low injury fraction.
- The violin plots show that most makes have a highly skewed distribution — most accidents result in zero fatal/serious injuries, but some accidents result in very high fractions.
- In terms of destruction rate, Maule, Dehavilland and Stinson show the lowest rates among small planes.

**Large Planes:**
- The large plane dataset is very limited (only 69 records with >20 passengers), making it difficult to make statistically robust claims.
- Boeing dominates the large plane dataset with sufficient records for comparison.

**Recommendations:**
- For small planes: Consider **Maule**, **Aviat Aircraft**, and **Cessna** — they combine low injury rates with large sample sizes giving statistical confidence.
- For large planes: **Boeing** shows consistently low injury rates based on available data.

### Analyze plane types (Make_Model)
- Plot the mean fatal/seriously injured fraction for both small and larger planes
- Filter ensuring at least ten individual examples per model

**Larger planes**

In [ ]:
large_model_stats = (
    df_large.groupby('Make_Model')
    .agg(mean_injury=('Fatal.Serious.Fraction', 'mean'), count=('Make_Model', 'count'))
    .query('count >= 10')
    .sort_values('mean_injury')
    .head(15)
)

if len(large_model_stats) > 0:
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.barh(large_model_stats.index, large_model_stats['mean_injury'], color='coral')
    ax.set_xlabel('Mean Fatal/Serious Injury Fraction')
    ax.set_title('Safest Large Plane Models (>=10 records, >20 passengers)')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()
    print(large_model_stats)
else:
    print('No large plane models have >= 10 records. Insufficient data for robust model-level analysis.')

**Smaller planes**
- Limit to makes with 10 lowest mean serious/fatal injury fractions

In [ ]:
# Filter to top 10 safest makes first
top10_small_makes = small_make_stats.head(10).index.tolist()
df_small_filtered = df_small[df_small['Make'].isin(top10_small_makes)]

small_model_stats = (
    df_small_filtered.groupby('Make_Model')
    .agg(mean_injury=('Fatal.Serious.Fraction', 'mean'), count=('Make_Model', 'count'))
    .query('count >= 10')
    .sort_values('mean_injury')
    .head(15)
)

fig, ax = plt.subplots(figsize=(14, 7))
ax.barh(small_model_stats.index, small_model_stats['mean_injury'], color='steelblue')
ax.set_xlabel('Mean Fatal/Serious Injury Fraction')
ax.set_title('Top 15 Safest Small Plane Models (from top 10 safest makes)')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

print(small_model_stats)

### Discussion of Specific Airplane Types

For small planes, several specific models stand out as having very low mean fatal/serious injury fractions. These models tend to be reliable single-engine aircraft with a long track record. The low injury fraction suggests that when accidents do occur, passengers are more likely to survive without fatal or serious injuries.

For large planes, the dataset is too small at the model level to make robust model-specific recommendations — most models have fewer than 10 accident records. Boeing as a make is the most data-rich option for large planes.

### Exploring Other Variables
Investigate how **Weather Condition** and **Number of Engines** affect aircraft damage and injury.

#### Factor 1: Weather Condition

We compare the mean fatal/serious injury fraction and destruction rate under VMC (Visual Meteorological Conditions) vs IMC (Instrument Meteorological Conditions).

In [ ]:
# Filter to VMC and IMC only
weather_df = df[df['Weather.Condition'].isin(['VMC', 'IMC'])].copy()

weather_stats = weather_df.groupby('Weather.Condition').agg(
    mean_injury=('Fatal.Serious.Fraction', 'mean'),
    mean_destroyed=('Destroyed', 'mean'),
    count=('Weather.Condition', 'count')
)

print(weather_stats)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(weather_stats.index, weather_stats['mean_injury'], color=['steelblue', 'coral'])
ax1.set_title('Mean Fatal/Serious Injury Fraction by Weather')
ax1.set_ylabel('Mean Fatal/Serious Injury Fraction')
ax1.set_xlabel('Weather Condition')

ax2.bar(weather_stats.index, weather_stats['mean_destroyed'], color=['steelblue', 'coral'])
ax2.set_title('Mean Destruction Rate by Weather')
ax2.set_ylabel('Mean Destruction Rate')
ax2.set_xlabel('Weather Condition')

plt.tight_layout()
plt.show()

**Discussion - Weather Condition:**

Accidents occurring under IMC (Instrument Meteorological Conditions — poor visibility, clouds, rain) show significantly higher fatal/serious injury fractions and destruction rates compared to VMC (clear visual conditions). This makes intuitive sense: flying in poor visibility is inherently more dangerous. 

This finding suggests that our client should weight their risk assessments by the typical operating conditions of the aircraft — planes that frequently operate in IMC conditions carry higher risk.

#### Factor 2: Number of Engines

We compare injury and destruction rates for single-engine vs twin-engine aircraft.

In [ ]:
# Filter to 1 or 2 engines (most common)
engine_df = df[df['Number.of.Engines'].isin([1.0, 2.0])].copy()
engine_df['Number.of.Engines'] = engine_df['Number.of.Engines'].astype(int).astype(str) + ' Engine(s)'

engine_stats = engine_df.groupby('Number.of.Engines').agg(
    mean_injury=('Fatal.Serious.Fraction', 'mean'),
    mean_destroyed=('Destroyed', 'mean'),
    count=('Number.of.Engines', 'count')
)

print(engine_stats)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

ax1.bar(engine_stats.index, engine_stats['mean_injury'], color=['steelblue', 'coral'])
ax1.set_title('Mean Fatal/Serious Injury Fraction by Number of Engines')
ax1.set_ylabel('Mean Fatal/Serious Injury Fraction')
ax1.set_xlabel('Number of Engines')

ax2.bar(engine_stats.index, engine_stats['mean_destroyed'], color=['steelblue', 'coral'])
ax2.set_title('Mean Destruction Rate by Number of Engines')
ax2.set_ylabel('Mean Destruction Rate')
ax2.set_xlabel('Number of Engines')

plt.tight_layout()
plt.show()

**Discussion - Number of Engines:**

Twin-engine aircraft show lower fatal/serious injury fractions and lower destruction rates compared to single-engine aircraft when accidents occur. This is consistent with the safety redundancy provided by having two engines — if one engine fails, the aircraft can still maintain some level of control with the other.

For the client, this suggests that two-engine aircraft represent a lower risk profile than single-engine aircraft, which is an important factor to consider when underwriting insurance policies.

## Summary of Recommendations

1. **For small aircraft**: Maule, Aviat Aircraft, Cessna, and Dehavilland show the best combination of low injury fractions and low destruction rates with statistically robust sample sizes.
2. **For large aircraft**: Boeing dominates the large aircraft accident data and shows low injury fractions — it is the most reliable recommendation for large aircraft.
3. **Weather**: VMC conditions are significantly safer than IMC — aircraft primarily used in VMC conditions pose lower risk.
4. **Engines**: Twin-engine aircraft are safer than single-engine aircraft in accident scenarios.